In [1]:
import torch
import torchvision
import torchvision.transforms as transforms

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225])
])

In [4]:
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
class DinosaurDataset(Dataset):
    def __init__(self, root_dir, transform = None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []

        for label in ["Theropod", "Sauropod", "Ornithischia", "Marine", "Unknown"]:
            class_dir = os.path.join(self.root_dir, label)
            for image_name in os.listdir(class_dir):
                self.image_paths.append(os.path.join(class_dir, image_name))
                self.labels.append(0 if label == "Theropod" else 1 if label == "Sauropod" else 1 if label == "Ornithischia" else 3 if label == "Marine" else 4 )
    
    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        label = self.labels[idx]

        return image, label

In [5]:
train_dataset = DinosaurDataset(root_dir = "dataset/train", transform = transform)
test_dataset = DinosaurDataset(root_dir = "dataset/test", transform = transform)

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle = True)
test_dataloader = DataLoader(test_dataset, batch_size = 32, shuffle = True)

In [6]:
import torchvision.models as models
import torch.nn as nn
model = models.resnet50(weights = models.ResNet50_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 5)

In [7]:
import torch.optim as optim
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr = 0.001, momentum = 0.9)

num_epochs = 10
from sklearn.metrics import accuracy_score

In [8]:
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_dataloader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()

        optimizer.step()

        running_loss += loss

    print(f"Epoch: {epoch +1}/{num_epochs}, Loss: {running_loss/len(train_dataloader)}")

    model.eval()


model.eval()
test_labels = []
test_preds = []
with torch.no_grad():
    for images, labels in test_dataloader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, preds =torch.max(outputs,1)

        test_labels.extend(labels.cpu().numpy())
        test_preds.extend(preds.cpu().numpy())
test_accuracy = accuracy_score(test_labels, test_preds)
print('Test Score: ', test_accuracy)


Epoch: 1/10, Loss: 0.9330446124076843
Epoch: 2/10, Loss: 0.48463135957717896
Epoch: 3/10, Loss: 0.280239999294281
Epoch: 4/10, Loss: 0.16603709757328033
Epoch: 5/10, Loss: 0.1008206382393837
Epoch: 6/10, Loss: 0.060482654720544815
Epoch: 7/10, Loss: 0.04303940758109093
Epoch: 8/10, Loss: 0.03384055569767952
Epoch: 9/10, Loss: 0.027905713766813278
Epoch: 10/10, Loss: 0.015644052997231483
Test Score:  0.8571428571428571


In [9]:
torch.save(model.state_dict(), "app/resnet_finetuned.pth")